# Monomer / dimer classification — Fig. 3c, 3d and S13A convolutional neural network labels each single-shot pnCCD diffractionpattern as one particle or two. The labels feed the monomer/dimer comparison ofFig. 3d; the confusion matrix is Fig. S13.The published model is `build_simplified_cnn`. `build_masked_cnn`, also presentin the original notebook, is a larger 3x3 / 32-64-128 variant that was trialledbut not used.

> **Data and weights.** Training needs the preprocessed pnCCD frames; classification> needs only the trained weights. Neither is in the repository — see the data DOI> in the README. Retraining does not reproduce the published weights bit for bit> (see "Reproducibility" in the README), so the deposited weights are the> authoritative artifact.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

from euxfel_nano import config
from euxfel_nano.seeds import set_all_seeds
from euxfel_nano.classifier import (build_simplified_cnn, preprocess_image,
                                    create_beamstop_mask, augment_img,
                                    process_convert_augment,
                                    evaluate_model_dual_threshold)

C = config.params()["classifier"]
rng = set_all_seeds(C["seed"])
for k, v in C.items():
    print(f"  {k}: {v}")

## InputsEach frame is mirrored to a square field, preserving symmetry and dynamicrange, normalized to [0, 1], and stacked with a fixed binary beam-stop maskinto a 200x200x2 tensor so the network learns only from informative pixels.Dimers are rare in the data, so the training set is augmented with aleft-right flip and four intensity-scaled copies (factors drawn uniformly from0.4-2.5), plus synthetic dimer patterns: the coherent sum of the far-fieldintensities of two spherical scatterers, with size, separation, azimuthalorientation and polarization angle drawn from distributions matching theexperiment, then normalized and given additive Gaussian noise. Syntheticdimers are used for training and validation only.

In [ ]:
# Replace with the paths to the preprocessed frames and labels.
patterns = xr.open_dataarray("diffraction_labels.nc")
simulated_dimers = np.load("./simulated_dimers.npy")

X, y = process_convert_augment(patterns, simulated_dimers)
print("input tensor:", X.shape, "  labels:", np.bincount(y.astype(int)))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=C["test_fraction"], stratify=y, random_state=C["seed"])
print("train:", X_train.shape[0], " test:", X_test.shape[0])

## TrainingA progressive scheme: the model is trained five times with decreasing learningrate, each stage admitting lower-intensity shots, so it first learns thehigh-signal patterns and is then fine-tuned on the harder ones. Class weightingcompensates the residual imbalance; the loss is binary focal loss withalpha = 0.8 and gamma = 2.0.

In [ ]:
model = build_simplified_cnn(tuple(C["input_shape"]))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=C["learning_rate"]),
    loss=tf.keras.losses.BinaryFocalCrossentropy(alpha=C["focal_alpha"],
                                                 gamma=C["focal_gamma"]),
    metrics=["accuracy",
             tf.keras.metrics.Recall(name="recall"),
             tf.keras.metrics.Precision(name="precision")])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=C["reduce_lr_on_plateau"]["factor"],
        patience=C["reduce_lr_on_plateau"]["patience"]),
    tf.keras.callbacks.EarlyStopping(
        patience=C["early_stopping"]["patience"],
        restore_best_weights=C["early_stopping"]["restore_best_weights"]),
]

n_pos, n_neg = int(y_train.sum()), int(len(y_train) - y_train.sum())
class_weight = {0: len(y_train) / (2 * n_neg), 1: len(y_train) / (2 * n_pos)}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=C["batch_size"],
    epochs=C["max_epochs"],
    class_weight=class_weight,
    callbacks=callbacks,
)

## Evaluation — Fig. S13Dual decision thresholds reject ambiguous shots: below 0.35 a frame is amonomer, above 0.65 a dimer, and anything between is left unclassified.

In [ ]:
th = C["decision_thresholds"]
evaluate_model_dual_threshold(model, X_test, y_test,
                              lower_threshold=th["monomer"],
                              upper_threshold=th["dimer"])

In [ ]:
p = model.predict(X_test).ravel()
keep = (p < th["monomer"]) | (p > th["dimer"])
y_hat = (p[keep] > th["dimer"]).astype(int)
y_true = y_test[keep].astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}   N={keep.sum()}")
print(f"precision {tp/(tp+fp):.3f}   recall {tp/(tp+fn):.3f}   "
      f"F1 {f1_score(y_true, y_hat):.3f}")

r = C["reported_confusion"]
print(f"\npublished (Fig. S13): TN={r['tn']} FP={r['fp']} FN={r['fn']} TP={r['tp']}"
      f"  -> precision 0.998, recall 0.989, F1 0.993")

In [ ]:
model.save("modelv5.keras")